# BDA Practica 2 — Knowledge Graph Exploration

This notebook explores the **RDF/RDFS Knowledge Graph** produced by the Exploitation Zone and runs the SPARQL analytical pipeline.

It is one of two analytical pipelines required by the P2 statement:

> Implement at least two Data Analysis pipelines:
> - 1 based on **pattern matching queries using SPARQL**
> - 1 based on running an **ML algorithm over the KG embeddings**

The first is showcased here. The second (KG embeddings + classifier) is showcased in `03_model_comparison_and_explainability.ipynb`.

Run the full pipeline (`python run_all_pipeline.py --skip-landing --strict`) before executing this notebook.

## 1. Setup and Manifest

Locate the project root, the KG artefacts and the manifest that links them.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from rdflib import Graph
import matplotlib.pyplot as plt


def locate_project_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(8):
        if (current / 'run_all_pipeline.py').exists():
            return current
        current = current.parent
    raise FileNotFoundError('Could not locate project root')

PROJECT_ROOT = locate_project_root(Path.cwd())
KG_ROOT = PROJECT_ROOT / 'Part4_Exploitation_zone' / 'exploitation_zone' / 'kg'
MANIFEST_PATH = KG_ROOT / 'kg_manifest.json'
REPORT_PATH = PROJECT_ROOT / 'Part5_Analysis_zone' / 'reports' / 'kg_analysis_report.json'

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
report = json.loads(REPORT_PATH.read_text(encoding='utf-8')) if REPORT_PATH.exists() else None

print('Manifest run_id        :', manifest['run_id'])
print('Created at (UTC)       :', manifest['created_at_utc'])
print('Language               :', manifest['language'])
print('Generator              :', manifest['generator'])
print('Trusted rows processed :', manifest['counts']['trusted_rows_processed'])
print('Health record resources:', manifest['counts']['health_record_resources'])
print('Aggregate measurements :', manifest['counts']['aggregate_measurements'])
print('Total triples (full)   :', manifest['counts']['triples'])


## 2. Load the Analytics Graph

The pipeline produces two Turtle files:
- `health_risk_kg.ttl` — the full graph with one `hr:HealthRecord` per cleaned trusted row.
- `health_risk_analytics_kg.ttl` — the compact graph with only schema, vocabulary and `hr:AggregateMeasurement` resources, much faster to query.

We use the analytics graph for interactive exploration.

In [ ]:
analytics_graph_path = Path(manifest['storage']['analytics_graph_file'])
analytics_graph_format = 'nt' if analytics_graph_path.suffix.lower() == '.nt' else 'turtle'

graph = Graph()
graph.parse(str(analytics_graph_path), format=analytics_graph_format)
print(f'Loaded analytics KG: {len(graph):,} triples from {analytics_graph_path}')


## 3. Graph Structure Summary

Quick counts of the main classes confirm the schema is populated correctly.

In [ ]:
STRUCTURE_QUERIES = {
    'Datasets': 'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a hr:Dataset }',
    'Age groups': 'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a hr:AgeGroup }',
    'Genders': 'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a hr:Gender }',
    'Population groups': 'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a hr:PopulationGroup }',
    'Indicators': 'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a hr:Indicator }',
    'Outcomes': 'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a hr:Outcome }',
    'Aggregate measurements': 'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a hr:AggregateMeasurement }',
}
prefix = 'PREFIX hr: <https://example.org/bda/health-risk/>'
rows = []
for label, q in STRUCTURE_QUERIES.items():
    result = list(graph.query(f'{prefix} {q}'))
    rows.append({'class': label, 'instances': int(result[0][0]) if result else 0})
structure_df = pd.DataFrame(rows)
structure_df

## 4. SPARQL Pattern-Matching Analysis

The Exploitation Zone writes a catalogue of `.rq` files in `exploitation_zone/kg/sparql/`. The analytical pipeline (`kg_analysis_pipeline.py`) runs a curated subset of them. Each query exploits the graph structure in a way that would be cumbersome to express over flat SQL tables.

In [ ]:
def query_to_df(graph: Graph, query_text: str) -> pd.DataFrame:
    results = graph.query(query_text)
    var_names = [str(v) for v in results.vars]
    data = []
    for row in results:
        record = {}
        for var in results.vars:
            value = row[var]
            if value is None:
                record[str(var)] = None
            elif hasattr(value, 'toPython'):
                record[str(var)] = value.toPython()
            else:
                record[str(var)] = str(value)
        data.append(record)
    return pd.DataFrame(data, columns=var_names)


SPARQL_DIR = Path(manifest['storage']['sparql_query_dir'])
for path in sorted(SPARQL_DIR.glob('*.rq')):
    print(path.name)

### 4.1 Indicators shared by multiple datasets

This query proves that the **common vocabulary** required by P2 is doing its job: the same `hr:Indicator` IRIs are referenced by aggregate measurements from more than one dataset.

In [ ]:
shared = query_to_df(graph, (SPARQL_DIR / '02_indicators_shared_by_datasets.rq').read_text())
shared = shared.sort_values('datasetCount', ascending=False)
shared

### 4.2 Highest-risk population groups

Rank `hr:PopulationGroup` × `hr:Dataset` pairs by the positive heart-disease rate of the aggregate measurement linked through `hr:hasOutcome`. We restrict to groups with at least 25 observations to drop noisy buckets.

In [ ]:
ranked = query_to_df(graph, (SPARQL_DIR / '03_rank_population_groups_by_outcome_rate.rq').read_text())
ranked = ranked.head(20)
ranked['shortGroup'] = ranked['populationGroup'].str.replace('https://example.org/bda/health-risk/', '', regex=False)
fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
ax.barh(ranked['shortGroup'] + '  (' + ranked['dataset'].str.split('/').str[-1] + ')',
        ranked['positiveRate'].astype(float),
        color='#C0504D')
ax.set_xlabel('Heart-disease positive rate')
ax.set_title('Top 20 population groups by outcome rate (n>=25)')
ax.invert_yaxis()
plt.show()
ranked[['shortGroup', 'dataset', 'positiveRate', 'observations']].head(10)

### 4.3 Combining indicators for the same population

For each group we read three aggregate measurements via `OPTIONAL` patterns. This is the kind of query a graph database makes natural — the equivalent SQL would require multiple self-joins.


In [ ]:
combined = query_to_df(graph, (SPARQL_DIR / '04_combine_indicators_same_group.rq').read_text())
for col in ['heartDiseaseRate', 'highBpRate', 'highCholesterolRate']:
    if col in combined.columns:
        combined[col] = pd.to_numeric(combined[col], errors='coerce')
combined.head(20)

In [ ]:
subset = combined.dropna(subset=['highBpRate'])
if not subset.empty:
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    for dataset, group in subset.groupby('dataset'):
        ax.scatter(group['highBpRate'], group['heartDiseaseRate'],
                   label=dataset.split('/')[-1], alpha=0.7)
    ax.set_xlabel('High blood pressure rate per group')
    ax.set_ylabel('Heart disease rate per group')
    ax.set_title('Population-level correlation between hypertension and outcome')
    ax.legend(loc='best')
    plt.show()
else:
    print('Not enough optional matches to plot.')

### 4.4 Indicator consistency across datasets

How stable is each indicator across the three sources? The query returns the min / max / average positive rate per boolean indicator. The largest spread highlights demographic or methodological differences between the sources.

In [ ]:
consistency = query_to_df(graph, (SPARQL_DIR / '06_indicator_consistency_across_datasets.rq').read_text())
for col in ['minRate', 'maxRate', 'avgRate']:
    consistency[col] = pd.to_numeric(consistency[col], errors='coerce')
consistency['spread'] = consistency['maxRate'] - consistency['minRate']
consistency.sort_values('spread', ascending=False)

### 4.5 Outcome rate by age group

Reuses the controlled `hr:AgeGroup` vocabulary to compare outcome rates across datasets and age bands.

In [ ]:
outcome_by_age = query_to_df(graph, (SPARQL_DIR / '07_outcome_rate_by_age_group.rq').read_text())
outcome_by_age['avgOutcomeRate'] = pd.to_numeric(outcome_by_age['avgOutcomeRate'], errors='coerce')
outcome_by_age['totalObservations'] = pd.to_numeric(outcome_by_age['totalObservations'], errors='coerce')

pivot = outcome_by_age.pivot_table(
    index='ageGroupLabel',
    columns=outcome_by_age['dataset'].str.split('/').str[-1],
    values='avgOutcomeRate',
)
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
pivot.plot(kind='bar', ax=ax)
ax.set_ylabel('Avg outcome positive rate')
ax.set_title('Heart-disease outcome rate by age group and source')
plt.xticks(rotation=30, ha='right')
plt.show()
pivot

### 4.6 Risk-factor co-occurrence in positive records (full graph)

This is the only query that benefits from the **full** KG (`health_risk_kg.ttl`) because it traverses `hr:hasObservedRiskFactor` on individual `hr:HealthRecord` resources. We load it lazily so the rest of the notebook stays light.

In [ ]:
full_graph_path = Path(manifest['storage']['graph_file'])
full_format = 'nt' if full_graph_path.suffix.lower() == '.nt' else 'turtle'
full_graph = Graph()
full_graph.parse(str(full_graph_path), format=full_format)
print(f'Loaded full KG: {len(full_graph):,} triples from {full_graph_path}')

In [ ]:
cooccurrence = query_to_df(full_graph, (SPARQL_DIR / '08_top_risk_factor_cooccurrence.rq').read_text())
cooccurrence['coOccurringRecords'] = pd.to_numeric(cooccurrence['coOccurringRecords'], errors='coerce')
cooccurrence

In [ ]:
if not cooccurrence.empty:
    cooccurrence['pair'] = cooccurrence['indicator1Label'] + ' & ' + cooccurrence['indicator2Label']
    fig, ax = plt.subplots(figsize=(8.5, 5.5), constrained_layout=True)
    ax.barh(cooccurrence['pair'].head(15)[::-1], cooccurrence['coOccurringRecords'].head(15)[::-1], color='#5C8FA1')
    ax.set_xlabel('Co-occurring heart-disease positive records')
    ax.set_title('Most frequent risk-factor pairs in positive records')
    plt.show()

## 5. Why a Knowledge Graph?

The same analyses on the raw SQL tables would require:

- Defining ad-hoc joins per question (population-group buckets, dataset codes, etc.).
- Hard-coding feature names (different per source) instead of reusing the canonical `hr:Indicator` IRIs.
- Carrying around a global join of all sources.

With the KG:

- Each cleaned trusted row becomes a `hr:HealthRecord` with explicit semantic links to `hr:Dataset`, `hr:PopulationGroup`, `hr:Outcome`, and indicator IRIs.
- Cross-dataset comparisons happen by reusing the same IRIs, not by joining columns.
- Adding a new dataset is one mapping in `exploitation_builders.py`; the SPARQL queries do not change.

## 6. Saved analytical report

The analytical pipeline persists a JSON report so the queries can be re-used by downstream tooling. The notebook merely re-executes them for visual exploration.

In [ ]:
if report is not None:
    print('Report timestamp:', report['run_at_utc'])
    print('Graph structure :', report['graph_structure'])
    print('Queries executed:', [q['query'] for q in report['queries']])
else:
    print('Run Part5_KG_Analysis first to generate the JSON report.')